# Gradient Boost and XGBoost

In this notebook, we will be exploring Gradient Boost and XGBoost using the existing Titanic dataset.

We will follow the basic preprocessing pipeline and train/validation split of the initial titanic.ipynb to ensure consistency.

We will then build a pipeline using *GradientBoostingClassifier*, then build a pipeline using *XGBClassifier*. Validation accuracies of XGB and Gradient Boost will be compared against earlier models to assess how well the new model performed.

This notebook aims to reinforce the learning of gradient boost and XGBoost. 

## 01. Load Data

In [1]:
# Import Modules
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
print("Set Up Complete")

# Load Data
titanic = pd.read_csv("train.csv")
titanic.head()

Set Up Complete


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## 02. Train/validation split, Basic Preprocessing Pipeline

Drop Passenger ID, Name, Ticket, Cabin columns
Separate X and y
Train/validation split
Fit preprocessing pipeline only on X_train

In [2]:
# Drop Passenger ID, Name, Ticket, Cabin
titanic_v2 = titanic.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"])

# Train/Test Split
from sklearn.model_selection import train_test_split
X = titanic_v2.drop(columns=["Survived"])
y = titanic_v2["Survived"]
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

# Separate numerical and categorical columns
numerical_columns = ["Pclass", "Age", "SibSp", "Parch", "Fare"]
categorical_columns = ["Sex", "Embarked"]

# Import necessary libraries
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

# Build preprocessing
numerical_transformer = Pipeline([("imputer", SimpleImputer(strategy="median"))])
categorical_transformer = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                                    ("encoder", OneHotEncoder())])

# Combine both
preprocessing = ColumnTransformer([("num", numerical_transformer, numerical_columns),
                                   ("cat", categorical_transformer, categorical_columns)],
                                   remainder = 'passthrough')

## 03. Gradient Boosting Pipeline

Random Forest builds many trees independently and combines their predictions, while Gradient Boost builds trees sequentially, each tree correcting the errors made by the existing ensemble. 

We will first keep the default settings to establish a baseline before experimenting with other parameters and hyperparameters.

In [4]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score

# Gradient Boosting Pipeline
gradient_boosting = Pipeline([("preprocessing", preprocessing), ("classifier", GradientBoostingClassifier(random_state=42))])

# Train model
gradient_boosting.fit(X_train, y_train)

# Predict on validation data
gradient_boosting_y_pred = gradient_boosting.predict(X_valid)

# Evaluate accuracy
accuracy_gradient_boosting = accuracy_score(y_valid, gradient_boosting_y_pred)
print(f"Gradient Boosting Accuracy: {accuracy_gradient_boosting:.4f}")

Gradient Boosting Accuracy: 0.8212


### 3.1 Experiment with n_estimators

In Gradient Boost, n_estimators controls the number of boosting stages. The default is 100, so we will explore different number of trees and check their validation accuracy scores.

From 50 to 200, adding boosting stages generally improves validation accuracy. Beyond 200, accuracy decreased slightly then levelled off.

In [7]:
n_estimator_values = [50, 100, 150, 200, 250, 300]
gradient_boosting_accuracies = []

for n in n_estimator_values:
    gradient_boosting = Pipeline([("preprocessing", preprocessing), ("classifier", GradientBoostingClassifier(n_estimators=n, random_state=42))])
    gradient_boosting.fit(X_train, y_train)
    gradient_boosting_y_pred = gradient_boosting.predict(X_valid)
    accuracy = accuracy_score(y_valid, gradient_boosting_y_pred)
    gradient_boosting_accuracies.append(accuracy)

for n, accuracy in zip(n_estimator_values, gradient_boosting_accuracies):
    print(f"n_estimators={n}: accuracy={accuracy:.4f}")


n_estimators=50: accuracy=0.7989
n_estimators=100: accuracy=0.8212
n_estimators=150: accuracy=0.8212
n_estimators=200: accuracy=0.8324
n_estimators=250: accuracy=0.8268
n_estimators=300: accuracy=0.8268


## 04. 